# 06 — Churn Feature Engineering

This notebook creates a time-safe customer snapshot. Every predictor is calculated from transactions on or before the snapshot date.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_loader import load_olist_data
from src.analytics import prepare_order_level
from src.churn import build_features_at_cutoff, build_churn_snapshots, FEATURE_COLUMNS

data = load_olist_data(ROOT / "data_raw")
order_level = prepare_order_level(data)
analysis_end = pd.Timestamp("2018-08-31 23:59:59")



## Feature families

- Recency and tenure: how recently and how long ago the customer started purchasing.
- Frequency: orders in the previous 30, 90 and 180 days and lifetime.
- Paid value: customer-paid value in the same periods.
- Basket behavior: item count, item price total and freight paid.
- Purchase rhythm: average and standard deviation of inter-purchase gaps.

These are behavioral predictors, not causal explanations.


In [ ]:
scoring_features = build_features_at_cutoff(
    order_level,
    cutoff_date=analysis_end,
    active_lookback_days=180
)

scoring_features[FEATURE_COLUMNS].describe().T



## Build historical training rows

A historical snapshot is retained only when its next 240 days are fully observable before the analysis end. The customer must have purchased in the preceding 180 days.


In [ ]:
training_snapshots = build_churn_snapshots(
    order_level,
    analysis_end=analysis_end,
    horizon_days=240,
    active_lookback_days=180,
    feature_lookback_days=180
)

training_snapshots.shape, training_snapshots["snapshot_date"].nunique()



In [ ]:
training_snapshots["churned_240d"].value_counts(dropna=False)



Before modeling, verify the target prevalence, snapshot dates, missing feature rates and that no feature uses future information. Generated snapshot tables are local artifacts and are not committed to Git.
